# 12. Interfacing with Compiled Code (Numba & C) (5+ Years Interview Guide)
Exhaustive revision guide to Python GIL bypass, JIT compilation with @jit(nopython=True), and high-speed loops on transaction arrays.

### Key 5-Year Interview Concepts Covered:
- **JIT Decoration (`@jit(nopython=True)`)**: Compiling Python numerical functions to native machine instructions.
- **Writing High-Performance Loops**: Executing path-dependent iterations at raw C speed.
- **Multi-Core Parallelization (`parallel=True`)**: Parallel loops with `prange()`.

This interactive revision guide loads and operates directly on `data/raw_transactions.csv` using dedicated cells per method.

In [ ]:
# Setup imports & dataset loading from raw_transactions.csv
import numpy as np
import pandas as pd
import sys
import time
import os

# Load raw transactions and extract aligned NumPy numeric arrays
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
raw_df = pd.read_csv(csv_path)
clean_raw = raw_df.dropna(subset=['transaction_amount', 'is_fraud', 'account_age_months']).reset_index(drop=True)
amounts = clean_raw['transaction_amount'].to_numpy(dtype=np.float64)
fraud_flags = clean_raw['is_fraud'].to_numpy(dtype=np.int8)
account_ages = clean_raw['account_age_months'].to_numpy(dtype=np.float32)

print(f"NumPy Version: {np.__version__}")
print(f"Loaded from {csv_path} ({len(amounts)} clean aligned rows):")
print(f"- amounts array: shape {amounts.shape}, dtype {amounts.dtype}")
print(f"- fraud_flags array: shape {fraud_flags.shape}, dtype {fraud_flags.dtype}")
print(f"- account_ages array: shape {account_ages.shape}, dtype {account_ages.dtype}")

### JIT Compilation with `@jit(nopython=True)`
**Explanation**: Compiles transaction sum of squares directly to LLVM machine code.

**Syntax**: `@jit(nopython=True) def fast_fn(arr): ...`

In [ ]:
try:
    from numba import jit
    @jit(nopython=True)
    def numba_sum_sq(arr):
        total = 0.0
        for i in range(len(arr)):
            total += arr[i] ** 2
        return total
    print('Numba JIT Sum of Squares on Transaction Amounts:', numba_sum_sq(amounts[:1000]))
except ImportError:
    print('Numba demo (install numba to test JIT compilation). Standard Python fallback shown.')

### Sequential Path-Dependent Exponential Moving Volatility
**Explanation**: Calculates iterative EWMA volatility where step $t$ depends strictly on step $t-1$ at raw C speed.

**Syntax**: `@jit(nopython=True) def ewma_vol(arr, alpha): ...`

In [ ]:
try:
    @jit(nopython=True)
    def ewma_vol(arr, alpha):
        n = len(arr)
        out = np.empty(n)
        out[0] = arr[0]
        for i in range(1, n):
            out[i] = alpha * arr[i] + (1.0 - alpha) * out[i-1]
        return out
    ewma_res = ewma_vol(amounts[:500], 0.05)
    print('EWMA Smoothed Amounts Head:', ewma_res[:5].round(2))
except NameError:
    pass

### Multi-Threaded Parallel Execution with `prange`
**Explanation**: Applies parallel multi-core thread execution on clean transaction rows.

**Syntax**: `@jit(nopython=True, parallel=True) def parallel_fn(arr): ...`

In [ ]:
try:
    from numba import prange
    @jit(nopython=True, parallel=True)
    def parallel_risk(arr, threshold):
        n = len(arr)
        flags = np.empty(n, dtype=np.int8)
        for i in prange(n):
            flags[i] = 1 if arr[i] > threshold else 0
        return flags
    p_flags = parallel_risk(amounts, 500.0)
    print('Parallel High-Value Flags Sum:', p_flags.sum())
except (ImportError, NameError):
    pass

## Section: Senior Fintech Interview Scenarios (5+ Years Experience)

### Q1: When to choose Numba JIT vs Pure NumPy Vectorization
**Explanation**: Explain trade-offs: Vectorization is ideal for batch operations; Numba is required for state-dependent time-series loops (EWMA, GARCH) without allocating huge temporary arrays.

**Syntax**: `@jit(nopython=True, fastmath=True)`

In [ ]:
print('Vectorization: Element-wise & matrix BLAS operations.')
print('Numba JIT: Recursive state-dependent loops without memory allocations.')